<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-11-self-hosting/lesson-11.1-gemma-cloud-run/practice/GCP_Capstone_11.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 11.1 — Gemma on Cloud Run L4

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: config, auth, and dependencies

Run this first. It installs the OpenAI SDK + Google auth libraries, authenticates via Application Default Credentials, and sets the deployment config every exercise below reuses. No API keys — the Cloud Run endpoint is private and called with an ID token.

In [ ]:
%%bash
pip install -q openai google-auth requests

In [ ]:
# Application Default Credentials (runs only on Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Colab auth OK — ADC ready')
except ImportError:
    print('Not on Colab — assuming ADC is already configured (gcloud auth application-default login)')

# Deployment config reused across every exercise
PROJECT_ID = 'documind-ai-YOUR-ID'
REGION     = 'us-central1'        # L4 GA region for the course
REPO       = 'vllm-repo'
IMAGE_NAME = 'gemma-3-4b-it'
SERVICE    = 'gemma-vllm'
MODEL      = 'google/gemma-3-4b-it'

# Filled in after deploy (Exercise 3). Placeholder until then.
SERVICE_URL = 'https://gemma-vllm-xxxxx-uc.a.run.app'

USD_INR = 85  # for INR cost displays

print('Deployment plan:')
print(f'  Project:  {PROJECT_ID}')
print(f'  Region:   {REGION} (L4 GA region)')
print(f'  Image:    {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{IMAGE_NAME}:latest')
print(f'  Service:  {SERVICE}')
print(f'  Model:    {MODEL}')

## Exercise 1: Dockerfile with Gemma 3 4B

**Difficulty:** Easy

Write a Dockerfile based on vllm/vllm-openai that downloads Gemma 3 4B-IT at build time with HF_TOKEN.

1. Base the image on `vllm/vllm-openai`.
2. Download `google/gemma-3-4b-it` at build time using an `HF_TOKEN` build arg.
3. Set `HF_HUB_OFFLINE=1` so the container makes no network calls at runtime.
4. Set the vLLM OpenAI server as the entrypoint with sensible flags.

**Expected behaviour:** Dockerfile with baked weights, `HF_HUB_OFFLINE=1`, vLLM entrypoint.

In [ ]:
DOCKERFILE = '''FROM vllm/vllm-openai:v0.16.0

ENV HF_HOME=/model-cache
ARG HF_TOKEN

# Bake weights into image for fast cold start
RUN huggingface-cli login --token ${HF_TOKEN} && \\
    huggingface-cli download google/gemma-3-4b-it

# Block runtime network calls
ENV HF_HUB_OFFLINE=1

EXPOSE 8080

ENTRYPOINT python3 -m vllm.entrypoints.openai.api_server \\
    --port ${PORT:-8080} \\
    --model google/gemma-3-4b-it \\
    --dtype bfloat16 \\
    --gpu-memory-utilization 0.90 \\
    --max-model-len 4096 \\
    --max-num-seqs 64 \\
    --cuda-graph-sizes 1,2,4,8,16,32,64 \\
    --enable-prefix-caching
'''

with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)
print('Dockerfile written')
print('Base: vllm/vllm-openai (~4.8GB)')
print('+ Gemma 3 4B weights (~8GB)')
print('= Final image ~13-15GB')

## Exercise 2: Build via Cloud Build

**Difficulty:** Easy

`cloudbuild.yaml` that pulls HF_TOKEN from Secret Manager and pushes the image to Artifact Registry.

1. Define a docker build step that reads `HF_TOKEN` from Secret Manager as a secret env.
2. Pass the token in as a `--build-arg`.
3. Push the image to Artifact Registry.
4. Use a big machine (`E2_HIGHCPU_32`) so the ~15GB build finishes in time.

**Expected behaviour:** E2_HIGHCPU_32 machine, secret mounted as build-arg, pushed to Artifact Registry.

In [ ]:
CLOUDBUILD = f'''steps:
- name: 'gcr.io/cloud-builders/docker'
  entrypoint: 'bash'
  secretEnv: ['HF_TOKEN_SECRET']
  args:
  - '-c'
  - |
      docker build \\
        --build-arg HF_TOKEN=$$HF_TOKEN_SECRET \\
        -t {REGION}-docker.pkg.dev/$PROJECT_ID/{REPO}/{IMAGE_NAME}:latest .
- name: 'gcr.io/cloud-builders/docker'
  args: ['push', '{REGION}-docker.pkg.dev/$PROJECT_ID/{REPO}/{IMAGE_NAME}:latest']
images: ['{REGION}-docker.pkg.dev/$PROJECT_ID/{REPO}/{IMAGE_NAME}:latest']
options:
  machineType: E2_HIGHCPU_32
availableSecrets:
  secretManager:
  - versionName: projects/$PROJECT_ID/secrets/HF_TOKEN/versions/latest
    env: 'HF_TOKEN_SECRET'
'''

with open('cloudbuild.yaml', 'w') as f:
    f.write(CLOUDBUILD)
print('cloudbuild.yaml written')
print()
print('Build command:')
print(f'  gcloud builds submit --config=cloudbuild.yaml --project={PROJECT_ID}')
print()
print('Expected build time: ~15-30 min on E2_HIGHCPU_32')

You can kick off the build from the shell (needs the `HF_TOKEN` secret and `vllm-repo` Artifact Registry repo to already exist):

In [ ]:
%%bash
# Create the Artifact Registry repo once (ignore error if it exists)
gcloud artifacts repositories create vllm-repo \
  --repository-format=docker \
  --location=us-central1 \
  --project=documind-ai-YOUR-ID || true

# Submit the build
gcloud builds submit --config=cloudbuild.yaml --project=documind-ai-YOUR-ID

## Exercise 3: Deploy with critical flags

**Difficulty:** Easy

`gcloud run deploy` with all three critical flags: `--no-cpu-throttling`, `--no-gpu-zonal-redundancy`, `--cpu-boost`.

1. Attach one `nvidia-l4` GPU.
2. Keep the service private (`--no-allow-unauthenticated`) and give it a dedicated service account.
3. Set a startup probe on `/health` (cold starts take up to ~2 min).
4. Include the three critical flags.

**Expected behaviour:** Full `gcloud run deploy` command with GPU, auth, startup probe.

In [ ]:
# PREREQUISITE (one-time, real GCP): enable the APIs, create the vLLM service account,
# store your Hugging Face token as a secret, and request L4 GPU quota (often 0 on new projects):
#   gcloud services enable run.googleapis.com cloudbuild.googleapis.com artifactregistry.googleapis.com secretmanager.googleapis.com
#   gcloud iam service-accounts create vllm-sa
#   printf '%s' "$HF_TOKEN" | gcloud secrets create HF_TOKEN --data-file=-
deploy_cmd = f'''gcloud run deploy {SERVICE} \\
  --image {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{IMAGE_NAME}:latest \\
  --region {REGION} \\
  --service-account vllm-sa@{PROJECT_ID}.iam.gserviceaccount.com \\
  --gpu 1 \\
  --gpu-type nvidia-l4 \\
  --cpu 8 \\
  --memory 32Gi \\
  --port 8000 \\
  --max-instances 3 \\
  --min-instances 0 \\
  --concurrency 64 \\
  --timeout 600 \\
  --no-cpu-throttling \\
  --no-gpu-zonal-redundancy \\
  --cpu-boost \\
  --no-allow-unauthenticated \\
  --startup-probe=httpGet.path=/health,httpGet.port=8000,initialDelaySeconds=120,failureThreshold=5,timeoutSeconds=10,periodSeconds=30
'''
print(deploy_cmd)
print()
print('THREE CRITICAL FLAGS:')
print('  --no-cpu-throttling        (vLLM needs CPU continuously, not just startup)')
print('  --no-gpu-zonal-redundancy  (cuts $1.047 to $0.672/hr = 40% savings)')
print('  --cpu-boost                (2x CPU during startup, cuts minutes off cold start)')

Run it for real (then capture the service URL into `SERVICE_URL` for the client exercises):

In [ ]:
%%bash
# PREREQUISITE (one-time, real GCP): enable the APIs, create the vLLM service account,
# store your Hugging Face token as a secret, and request L4 GPU quota (often 0 on new projects):
#   gcloud services enable run.googleapis.com cloudbuild.googleapis.com artifactregistry.googleapis.com secretmanager.googleapis.com
#   gcloud iam service-accounts create vllm-sa
#   printf '%s' "$HF_TOKEN" | gcloud secrets create HF_TOKEN --data-file=-
gcloud run deploy gemma-vllm \
  --image us-central1-docker.pkg.dev/documind-ai-YOUR-ID/vllm-repo/gemma-3-4b-it:latest \
  --region us-central1 \
  --service-account vllm-sa@documind-ai-YOUR-ID.iam.gserviceaccount.com \
  --gpu 1 --gpu-type nvidia-l4 --cpu 8 --memory 32Gi --port 8000 \
  --max-instances 3 --min-instances 0 --concurrency 64 --timeout 600 \
  --no-cpu-throttling --no-gpu-zonal-redundancy --cpu-boost \
  --no-allow-unauthenticated \
  --startup-probe=httpGet.path=/health,httpGet.port=8000,initialDelaySeconds=120,failureThreshold=5,timeoutSeconds=10,periodSeconds=30

# Grab the URL:
gcloud run services describe gemma-vllm --region us-central1 \
  --format='value(status.url)' --project=documind-ai-YOUR-ID

In [ ]:
# Paste the URL printed above so the client exercises can reach the service.
# SERVICE_URL = 'https://gemma-vllm-xxxxx-uc.a.run.app'
print('SERVICE_URL is currently:', SERVICE_URL)
print('Edit the line above with the real URL from the deploy step before running Exercises 4-6.')

## Exercise 4: Python client with auth

**Difficulty:** Medium

OpenAI SDK + `google.oauth2.id_token.fetch_id_token()` to call the private Cloud Run endpoint.

1. Fetch an ID token scoped to the service URL.
2. Build an `OpenAI` client pointed at `{service_url}/v1` with the token in an Authorization header.
3. Add a DocuMind classification helper.
4. Add a streaming Q&A helper.

**Expected behaviour:** OpenAI client with Authorization header containing an ID token.

In [ ]:
import google.oauth2.id_token
import google.auth.transport.requests
from openai import OpenAI

def get_authenticated_client(service_url: str) -> OpenAI:
    """Returns an OpenAI client configured for a private Cloud Run service."""
    request = google.auth.transport.requests.Request()
    id_token = google.oauth2.id_token.fetch_id_token(request, service_url)
    return OpenAI(
        base_url=f"{service_url}/v1",
        api_key="not-needed",
        default_headers={"Authorization": f"Bearer {id_token}"},
    )

# DocuMind classification example
def classify_document(text: str) -> str:
    client = get_authenticated_client(SERVICE_URL)
    response = client.chat.completions.create(
        model="google/gemma-3-4b-it",
        messages=[
            {"role": "system", "content": "Classify as INVOICE, CONTRACT, REPORT, or RECEIPT. Reply ONE word."},
            {"role": "user", "content": text},
        ],
        max_tokens=10,
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

# DocuMind streaming Q&A
def stream_answer(question: str):
    client = get_authenticated_client(SERVICE_URL)
    stream = client.chat.completions.create(
        model="google/gemma-3-4b-it",
        messages=[{"role": "user", "content": question}],
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)

print('Client helpers ready. Once SERVICE_URL is set, try:')
print('  classify_document("Invoice #4471, amount due Rs 12,500, net 30 days")')
print('  stream_answer("Summarise a net-30 payment term in one line")')

## Exercise 5: Health check poller

**Difficulty:** Medium

Poll the `/health` endpoint with exponential backoff before the first inference (for cold starts).

1. Send an authenticated GET to `/health`.
2. Back off exponentially between attempts, capped at 30s.
3. Return `True` on a 200, `False` after max attempts.
4. Call it before the first request when `min-instances=0`.

**Expected behaviour:** `wait_for_ready()` with exponential backoff capped at 30s.

In [ ]:
import time
import requests

def wait_for_ready(service_url: str, id_token: str,
                   max_attempts: int = 20, base_delay: float = 2.0) -> bool:
    """Exponential backoff health poll.

    Gemma 3 4B cold start ~19s. Gemma 9B ~35s. This poller handles both.
    Returns True when /health returns 200, False after max_attempts.
    """
    headers = {'Authorization': f'Bearer {id_token}'}
    for attempt in range(max_attempts):
        try:
            r = requests.get(f'{service_url}/health', headers=headers, timeout=5)
            if r.status_code == 200:
                print(f'  Ready after {attempt + 1} attempts')
                return True
        except requests.RequestException:
            pass
        delay = min(base_delay * (1.5 ** attempt), 30.0)
        print(f'  Attempt {attempt + 1}: not ready, waiting {delay:.1f}s')
        time.sleep(delay)
    return False

print('wait_for_ready() ready')
print('Call BEFORE first request when min-instances=0 to avoid timeouts during cold start')

# Usage once SERVICE_URL is set:
# req = google.auth.transport.requests.Request()
# tok = google.oauth2.id_token.fetch_id_token(req, SERVICE_URL)
# wait_for_ready(SERVICE_URL, tok)

## Exercise 6: Cold start measurement

**Difficulty:** Medium

Log time-to-first-token across 10 cold starts. Compare before/after the `--cuda-graph-sizes` limit.

1. For each trial, wait for `/health`, then time how long until the first streamed token arrives.
2. Force a fresh instance between trials (scale to zero / idle out) so each is a genuine cold start.
3. Record TTFT for all 10 trials and report min/median/max.
4. Compare the median against the no-limit baseline.

**Expected behaviour:** ~54s without limit, ~15s with `--cuda-graph-sizes 1,2,4,8,16,32,64`.

In [ ]:
# NOTE: written fresh for this lab (the lesson notebook describes the numbers
# but does not ship a TTFT harness). It reuses wait_for_ready + get_authenticated_client.
import time
import statistics
import google.oauth2.id_token
import google.auth.transport.requests

def time_to_first_token(service_url: str) -> float:
    """Cold-start TTFT: token 0 arrival measured from a fresh /health-ready instance."""
    req = google.auth.transport.requests.Request()
    tok = google.oauth2.id_token.fetch_id_token(req, service_url)

    # Wait for the (cold) instance to report healthy before timing generation.
    wait_for_ready(service_url, tok)

    client = get_authenticated_client(service_url)
    start = time.time()
    stream = client.chat.completions.create(
        model="google/gemma-3-4b-it",
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=5,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            return time.time() - start  # first token seen
    return time.time() - start

def measure_cold_starts(service_url: str, trials: int = 10,
                        idle_seconds: int = 900) -> list:
    """Run `trials` cold starts. Between trials the service must scale to zero
    (min-instances=0). idle_seconds is how long Cloud Run keeps a GPU instance
    warm before it idles out — sleep past it for a true cold start."""
    ttfts = []
    for i in range(trials):
        ttft = time_to_first_token(service_url)
        ttfts.append(ttft)
        print(f'  Trial {i+1:2d}: TTFT = {ttft:5.1f}s')
        if i < trials - 1:
            print(f'  Idling {idle_seconds}s so the next start is cold...')
            time.sleep(idle_seconds)
    print()
    print(f'  min={min(ttfts):.1f}s  median={statistics.median(ttfts):.1f}s  max={max(ttfts):.1f}s')
    return ttfts

print('measure_cold_starts() ready.')
print('Expected median with --cuda-graph-sizes 1,2,4,8,16,32,64: ~15s')
print('Expected median WITHOUT the limit (full graph capture): ~54s')
print()
print('Run when SERVICE_URL is live:  ttfts = measure_cold_starts(SERVICE_URL, trials=10)')

## Exercise 7: Cost calculator

**Difficulty:** Challenge

Python function comparing Cloud Run L4 vs Gemini Flash vs Pro at 2M / 10M / 50M / 100M tokens/day.

1. Model the Cloud Run instance cost (L4 GPU + CPU + memory per hour).
2. Model Gemini Flash and Pro cost with an 80% input / 20% output split.
3. Scale Cloud Run instances above ~25M tokens/day.
4. Print the break-even point.

**Expected behaviour:** Break-even ~4M tokens/day vs Flash. Cloud Run beats Pro above ~3M tokens/day (up to ~9x at 50M).

In [ ]:
# Standard pricing (per 1M tokens, USD)
L4_GPU_HR    = 0.672    # no zonal redundancy
CPU_8VCPU_HR = 0.518
MEM_32GIB_HR = 0.230
INSTANCE_HR  = L4_GPU_HR + CPU_8VCPU_HR + MEM_32GIB_HR  # ~$1.42/hr

# Gemini API pricing (per 1M tokens)
FLASH_IN, FLASH_OUT = 1.50, 7.50
PRO_IN,   PRO_OUT   = 2.00, 12.00

def cloud_run_cost(tokens_per_day: int, hours_per_day: int = 8,
                   instances_needed: int = 1):
    """Monthly cost of self-hosting Gemma on Cloud Run L4."""
    days_per_month = 30
    hrs_per_month = hours_per_day * days_per_month * instances_needed
    return INSTANCE_HR * hrs_per_month

def gemini_cost(tokens_per_day: int, model: str = 'flash'):
    """Monthly Gemini API cost assuming 80% input, 20% output split."""
    days_per_month = 30
    total = tokens_per_day * days_per_month
    input_tokens  = total * 0.8
    output_tokens = total * 0.2
    if model == 'flash':
        return (input_tokens * FLASH_IN + output_tokens * FLASH_OUT) / 1_000_000
    elif model == 'pro':
        return (input_tokens * PRO_IN + output_tokens * PRO_OUT) / 1_000_000

print(f'{"Volume":<15} {"Cloud Run L4":>15} {"Gemini Flash":>15} {"Gemini Pro":>15}')
print('-' * 65)
for volume in [2_000_000, 10_000_000, 50_000_000, 100_000_000]:
    instances = max(1, volume // 25_000_000)  # scale for >25M tokens/day
    cr = cloud_run_cost(volume, hours_per_day=8, instances_needed=instances)
    gf = gemini_cost(volume, 'flash')
    gp = gemini_cost(volume, 'pro')
    vol_str = f'{volume // 1_000_000}M/day'
    print(f'{vol_str:<15} ${cr:>12,.0f} ${gf:>12,.0f} ${gp:>12,.0f}')

print()
print('Break-even: Cloud Run beats Gemini Flash around ~4M tokens/day')
print('Cloud Run beats Gemini Pro above ~3M tokens/day. Up to ~9x cheaper at 50M.')
print()
# INR view of the ~$1.42/hr instance
print(f'Instance rate: ${INSTANCE_HR:.2f}/hr  =  Rs {INSTANCE_HR * USD_INR:,.0f}/hr  (USD_INR={USD_INR})')

## Exercise 8: Multi-model constellation

**Difficulty:** Challenge

Deploy 3 Gemma models (2B INT4 + 4B BF16 + 9B INT8) on one L4 with separate vLLM server processes.

1. Assign each model its own port and quantization.
2. Keep total VRAM under the 24GB the L4 provides.
3. Route DocuMind stages (classify -> extract -> summarize) to the right port.
4. Note the tradeoffs vs three separate endpoints.

**Expected behaviour:** Three vLLM processes on different ports, total VRAM ~19GB.

In [ ]:
# Pattern 3 from the lesson: three vLLM servers sharing ONE L4.
PATTERN_3 = {
    'name': 'Multi-model constellation on ONE L4',
    'use_case': 'DocuMind pipeline: classify -> extract -> summarize',
    'models': {
        'classifier':  'Gemma 2B FP8  (~2.5GB)',
        'extractor':   'Gemma 3 4B BF16 (~8GB)',
        'summarizer':  'Gemma 2 9B FP8 (~9GB)',
    },
    'total_vram': '~20GB on 24GB L4 (plenty of KV cache)',
    'cost_profile': '~$1.42/hr ONE instance vs 3 separate Vertex AI endpoints',
    'tradeoff': 'Single point of failure, more complex routing, massive cost savings',
}

for k, v in PATTERN_3.items():
    print(f'{k}: {v}')

The three processes are launched inside one container, each pinned to its own port and a slice of GPU memory. Below is the entrypoint pattern (written for this lab, extending the Dockerfile from Exercise 1) plus a router that sends each DocuMind stage to the right port.

In [ ]:
# Startup script: three vLLM OpenAI servers on ports 8001/8002/8003, one L4.
# gpu-memory-utilization is fractioned so the three fit inside 24GB.
# All three use on-the-fly fp8 (dynamic) quantization — no pre-quantized
# checkpoint needed. (AWQ would require a pre-quantized AWQ repo, which
# google/gemma-2b-it is not, so vLLM would fail to start with --quantization awq.)
CONSTELLATION_ENTRYPOINT = '''#!/bin/bash
set -e

# Classifier: Gemma 2B, FP8, small slice
python3 -m vllm.entrypoints.openai.api_server \\
  --port 8001 --model google/gemma-2b-it --quantization fp8 \\
  --gpu-memory-utilization 0.15 --max-model-len 2048 &

# Extractor: Gemma 3 4B, BF16
python3 -m vllm.entrypoints.openai.api_server \\
  --port 8002 --model google/gemma-3-4b-it --dtype bfloat16 \\
  --gpu-memory-utilization 0.38 --max-model-len 4096 &

# Summarizer: Gemma 2 9B, FP8
python3 -m vllm.entrypoints.openai.api_server \\
  --port 8003 --model google/gemma-2-9b-it --quantization fp8 \\
  --gpu-memory-utilization 0.40 --max-model-len 4096 &

wait -n  # exit if any server dies
'''

with open('constellation_entrypoint.sh', 'w') as f:
    f.write(CONSTELLATION_ENTRYPOINT)
print('constellation_entrypoint.sh written (~20GB total VRAM on a 24GB L4)')

# DocuMind stage router: each stage -> its own port + model.
STAGE_PORTS = {
    'classify':  (8001, 'google/gemma-2b-it'),
    'extract':   (8002, 'google/gemma-3-4b-it'),
    'summarize': (8003, 'google/gemma-2-9b-it'),
}

def stage_client(service_url: str, stage: str) -> OpenAI:
    """Authenticated OpenAI client pointed at the vLLM process for `stage`."""
    port, _ = STAGE_PORTS[stage]
    request = google.auth.transport.requests.Request()
    id_token = google.oauth2.id_token.fetch_id_token(request, service_url)
    return OpenAI(
        base_url=f"{service_url}:{port}/v1",
        api_key="not-needed",
        default_headers={"Authorization": f"Bearer {id_token}"},
    )

def run_pipeline(service_url: str, text: str) -> dict:
    """classify -> extract -> summarize across the three co-located models."""
    out = {}
    for stage in ('classify', 'extract', 'summarize'):
        _, model = STAGE_PORTS[stage]
        client = stage_client(service_url, stage)
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": f"[{stage}] {text}"}],
            max_tokens=128,
        )
        out[stage] = resp.choices[0].message.content.strip()
    return out

print('Router ready:  run_pipeline(SERVICE_URL, "<document text>")')